# Plant Disease Object Detection Model Training
## YOLOv8 Object Detection on PlantDoc Dataset

This notebook trains a YOLOv8 object detection model to:
- **Detect** disease locations in leaf images
- **Localize** bounding boxes around diseased regions
- **Classify** disease types with confidence scores

**Dataset:** PlantDoc-Object-Detection-Dataset with ~27 disease classes and bounding box annotations

## 1. Install Dependencies

In [1]:
import subprocess
import sys

print("Installing YOLOv8 and dependencies...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyyaml"])
print("YOLOv8 installed successfully!")

Installing YOLOv8 and dependencies...
YOLOv8 installed successfully!


## 2. Import Libraries

In [2]:
import os
import json
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
from shutil import copy2
from ultralytics import YOLO
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\Siri Nenta\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
All libraries imported successfully!


## 3. Dataset Configuration

In [3]:
# Dataset paths
OBJ_DET_BASE_PATH = Path('c:/Users/Siri Nenta/OneDrive/Desktop/agri/PlantDoc-Object-Detection-Dataset')
OBJ_DET_TRAIN_CSV = OBJ_DET_BASE_PATH / 'train_labels.csv'
OBJ_DET_TEST_CSV = OBJ_DET_BASE_PATH / 'test_labels.csv'
OBJ_DET_TRAIN_IMG = OBJ_DET_BASE_PATH / 'TRAIN'
OBJ_DET_TEST_IMG = OBJ_DET_BASE_PATH / 'TEST' / 'TEST'

# Output configuration
OUTPUT_DIR = Path('models')
OUTPUT_DIR.mkdir(exist_ok=True)
OBJ_DET_OUTPUT_DIR = OUTPUT_DIR / 'object_detection'
OBJ_DET_OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Object Detection Dataset Path: {OBJ_DET_BASE_PATH}")
print(f"Train CSV exists: {OBJ_DET_TRAIN_CSV.exists()}")
print(f"Test CSV exists: {OBJ_DET_TEST_CSV.exists()}")
print(f"Train Images dir exists: {OBJ_DET_TRAIN_IMG.exists()}")
print(f"Test Images dir exists: {OBJ_DET_TEST_IMG.exists()}")

# Load CSV files
train_labels_df = pd.read_csv(OBJ_DET_TRAIN_CSV)
test_labels_df = pd.read_csv(OBJ_DET_TEST_CSV)

print(f"\nTrain labels shape: {train_labels_df.shape}")
print(f"Test labels shape: {test_labels_df.shape}")
print(f"\nDisease classes: {train_labels_df['class'].unique()[:10]}...")  # Show first 10
print(f"Number of unique disease classes: {train_labels_df['class'].nunique()}")

Object Detection Dataset Path: c:\Users\Siri Nenta\OneDrive\Desktop\agri\PlantDoc-Object-Detection-Dataset
Train CSV exists: True
Test CSV exists: True
Train Images dir exists: True
Test Images dir exists: True

Train labels shape: (8469, 8)
Test labels shape: (452, 8)

Disease classes: <StringArray>
[             'Cherry leaf',               'Peach leaf',
         'Corn leaf blight',          'Apple rust leaf',
  'Potato leaf late blight',          'Strawberry leaf',
           'Corn rust leaf',  'Tomato leaf late blight',
         'Tomato mold leaf', 'Potato leaf early blight']
Length: 10, dtype: str...
Number of unique disease classes: 29


## 4. Prepare YOLO Dataset Format

In [4]:
# Create YOLO directory structure
yolo_dataset_dir = OBJ_DET_OUTPUT_DIR / 'dataset'
yolo_dataset_dir.mkdir(exist_ok=True)

yolo_images_train = yolo_dataset_dir / 'images' / 'train'
yolo_images_val = yolo_dataset_dir / 'images' / 'val'
yolo_labels_train = yolo_dataset_dir / 'labels' / 'train'
yolo_labels_val = yolo_dataset_dir / 'labels' / 'val'

for d in [yolo_images_train, yolo_images_val, yolo_labels_train, yolo_labels_val]:
    d.mkdir(parents=True, exist_ok=True)

print("YOLO dataset directory structure created")

# Get unique classes and create mapping
disease_classes = sorted(train_labels_df['class'].unique())
class_to_idx = {cls: idx for idx, cls in enumerate(disease_classes)}
idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}

print(f"\nDisease Classes ({len(disease_classes)}):")
for idx, cls in idx_to_class.items():
    print(f"  {idx:2d}: {cls}")

# Save class mapping
class_mapping_yolo = {idx: cls for cls, idx in class_to_idx.items()}
with open(yolo_dataset_dir / 'classes.json', 'w') as f:
    json.dump(class_mapping_yolo, f, indent=2)

print("\nClass mapping saved")

YOLO dataset directory structure created

Disease Classes (29):
   0: Apple Scab Leaf
   1: Apple leaf
   2: Apple rust leaf
   3: Bell_pepper leaf
   4: Bell_pepper leaf spot
   5: Blueberry leaf
   6: Cherry leaf
   7: Corn Gray leaf spot
   8: Corn leaf blight
   9: Corn rust leaf
  10: Peach leaf
  11: Potato leaf
  12: Potato leaf early blight
  13: Potato leaf late blight
  14: Raspberry leaf
  15: Soyabean leaf
  16: Squash Powdery mildew leaf
  17: Strawberry leaf
  18: Tomato Early blight leaf
  19: Tomato Septoria leaf spot
  20: Tomato leaf
  21: Tomato leaf bacterial spot
  22: Tomato leaf late blight
  23: Tomato leaf mosaic virus
  24: Tomato leaf yellow virus
  25: Tomato mold leaf
  26: Tomato two spotted spider mites leaf
  27: grape leaf
  28: grape leaf black rot

Class mapping saved


## 5. Convert Annotations to YOLO Format

In [7]:
print("Converting annotations to YOLO format and organizing dataset...")
print("This may take a few minutes for large datasets...")

# Process training data
train_processed = []
for idx, row in train_labels_df.iterrows():
    img_filename = row['filename']
    src_img = OBJ_DET_TRAIN_IMG / img_filename
    
    if src_img.exists():
        # Copy image to train folder
        dst_img = yolo_images_train / img_filename
        if not dst_img.exists():
            copy2(str(src_img), str(dst_img))
        
        # Create YOLO annotation
        img_width = row['width']
        img_height = row['height']
        
        # Skip invalid annotations (width/height = 0 or NaN)
        if pd.isna(img_width) or pd.isna(img_height) or img_width <= 0 or img_height <= 0:
            continue
        
        class_id = class_to_idx[row['class']]
        
        # Convert bbox to YOLO format (normalized center x, center y, width, height)
        xmin, ymin, xmax, ymax = row['xmin'], row['ymin'], row['xmax'], row['ymax']
        center_x = (xmin + xmax) / 2 / img_width
        center_y = (ymin + ymax) / 2 / img_height
        bbox_width = (xmax - xmin) / img_width
        bbox_height = (ymax - ymin) / img_height
        
        train_processed.append({
            'image': img_filename,
            'class_id': class_id,
            'center_x': center_x,
            'center_y': center_y,
            'width': bbox_width,
            'height': bbox_height
        })

# Process test data as validation
val_processed = []
for idx, row in test_labels_df.iterrows():
    img_filename = row['filename']
    src_img = OBJ_DET_TEST_IMG / img_filename
    
    if src_img.exists():
        # Copy image to val folder
        dst_img = yolo_images_val / img_filename
        if not dst_img.exists():
            copy2(str(src_img), str(dst_img))
        
        # Create YOLO annotation
        img_width = row['width']
        img_height = row['height']
        
        # Skip invalid annotations (width/height = 0 or NaN)
        if pd.isna(img_width) or pd.isna(img_height) or img_width <= 0 or img_height <= 0:
            continue
        
        class_id = class_to_idx[row['class']]
        
        xmin, ymin, xmax, ymax = row['xmin'], row['ymin'], row['xmax'], row['ymax']
        center_x = (xmin + xmax) / 2 / img_width
        center_y = (ymin + ymax) / 2 / img_height
        bbox_width = (xmax - xmin) / img_width
        bbox_height = (ymax - ymin) / img_height
        
        val_processed.append({
            'image': img_filename,
            'class_id': class_id,
            'center_x': center_x,
            'center_y': center_y,
            'width': bbox_width,
            'height': bbox_height
        })

# Write YOLO label files
for item in train_processed:
    label_file = yolo_labels_train / (Path(item['image']).stem + '.txt')
    with open(label_file, 'a') as f:
        f.write(f"{item['class_id']} {item['center_x']:.6f} {item['center_y']:.6f} {item['width']:.6f} {item['height']:.6f}\n")

for item in val_processed:
    label_file = yolo_labels_val / (Path(item['image']).stem + '.txt')
    with open(label_file, 'a') as f:
        f.write(f"{item['class_id']} {item['center_x']:.6f} {item['center_y']:.6f} {item['width']:.6f} {item['height']:.6f}\n")

print(f"\nTraining samples: {len(set([p['image'] for p in train_processed]))}")
print(f"Validation samples: {len(set([p['image'] for p in val_processed]))}")
print(f"Total annotations: {len(train_processed) + len(val_processed)}")

Converting annotations to YOLO format and organizing dataset...
This may take a few minutes for large datasets...

Training samples: 2249
Validation samples: 231
Total annotations: 8585


## 6. Create YOLO Configuration

In [8]:
# Create YOLO dataset.yaml configuration file
yolo_config = {
    'path': str(yolo_dataset_dir),
    'train': str(yolo_images_train.relative_to(yolo_dataset_dir)),
    'val': str(yolo_images_val.relative_to(yolo_dataset_dir)),
    'nc': len(disease_classes),
    'names': disease_classes
}

yolo_yaml_path = yolo_dataset_dir / 'data.yaml'
with open(yolo_yaml_path, 'w') as f:
    yaml.dump(yolo_config, f, default_flow_style=False)

print("YOLO configuration file created:")
print(f"  Path: {yolo_yaml_path}")
print(f"  Number of classes: {yolo_config['nc']}")
print(f"  Classes: {', '.join(yolo_config['names'][:5])}...")

YOLO configuration file created:
  Path: models\object_detection\dataset\data.yaml
  Number of classes: 29
  Classes: Apple Scab Leaf, Apple leaf, Apple rust leaf, Bell_pepper leaf, Bell_pepper leaf spot...


## 7. Train YOLOv8 Model

In [ ]:
print("Training YOLOv8 Object Detection Model...")
print("="*70)

# Load YOLOv8 model (nano for faster training, can change to small/medium/large)
yolo_model = YOLO('yolov8n.pt')

# Train the model
obj_det_results = yolo_model.train(
    data=str(yolo_yaml_path),
    epochs=50,
    imgsz=416,
    batch=16,
    patience=10,
    device='cpu',  # Use 'cpu' for CPU, or device=0 if GPU available
    project=str(OBJ_DET_OUTPUT_DIR),
    name='yolov8_disease_detection',
    exist_ok=True,
    verbose=True
)

print("\nObject Detection Training completed!")
print(f"Best model saved at: {OBJ_DET_OUTPUT_DIR / 'yolov8_disease_detection/weights/best.pt'}")

Training YOLOv8 Object Detection Model...
Ultralytics 8.4.30  Python-3.13.11 torch-2.10.0+cpu CPU (13th Gen Intel Core i7-13620H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=models\object_detection\dataset\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8_disease_detection, nbs=64, nms=False, opset=None, 

## 8. Evaluate Model

In [ ]:
print("Evaluating Object Detection Model...")
print("="*70)

# Load the best trained model
best_model = YOLO(str(OBJ_DET_OUTPUT_DIR / 'yolov8_disease_detection/weights/best.pt'))

# Validate the model
val_results = best_model.val()

print(f"\nValidation Results:")
print(f"  Box (IoU=0.50): {val_results.box.map50:.4f}")
print(f"  Box (IoU=0.50:0.95): {val_results.box.map:.4f}")
print(f"  Precision: {val_results.box.mp:.4f}")
print(f"  Recall: {val_results.box.mr:.4f}")

## 9. Test Inference

In [ ]:
print("Testing inference on sample validation images...\n")
test_image_paths = list(yolo_images_val.glob('*.jpg'))[:5]

for img_path in test_image_paths:
    results = best_model.predict(source=str(img_path), conf=0.5, verbose=False)
    num_detections = len(results[0].boxes)
    print(f"  {img_path.name}: {num_detections} detections found")

## 10. Visualize Results

In [ ]:
print("Visualizing Object Detection Results...\n")

# Run inference on a sample image
sample_images = list(yolo_images_val.glob('*.jpg'))
if sample_images:
    sample_image = sample_images[0]
    results = best_model.predict(source=str(sample_image), conf=0.5, save=False, verbose=False)
    
    # Plot results
    fig, axes = plt.subplots(1, 1, figsize=(12, 8))
    
    # Load and display the image
    img = cv2.imread(str(sample_image))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Draw bounding boxes
    for box in results[0].boxes:
        x1, y1, x2, y2 = box.xyxy[0]
        conf = box.conf[0]
        cls_id = int(box.cls[0])
        cls_name = disease_classes[cls_id]
        
        # Draw rectangle
        cv2.rectangle(img_rgb, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        
        # Add label
        label = f"{cls_name} {conf:.2f}"
        cv2.putText(img_rgb, label, (int(x1), int(y1)-10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    
    axes.imshow(img_rgb)
    axes.set_title(f'Object Detection Results - {sample_image.name}', fontweight='bold', fontsize=12)
    axes.axis('off')
    
    plt.tight_layout()
    plt.savefig(str(OBJ_DET_OUTPUT_DIR / 'sample_detection.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    print("Sample detection visualization saved!")
else:
    print("No images found in validation set")

## 11. Model Summary and Export

In [ ]:
# Save Object Detection Model Summary
obj_det_summary = {
    'model_type': 'YOLOv8 Nano',
    'task': 'Object Detection',
    'dataset': 'PlantDoc-Object-Detection-Dataset',
    'num_classes': len(disease_classes),
    'classes': disease_classes,
    'training_images': len(set([p['image'] for p in train_processed])),
    'validation_images': len(set([p['image'] for p in val_processed])),
    'total_annotations': len(train_processed) + len(val_processed),
    'epochs': 50,
    'image_size': 416,
    'batch_size': 16,
    'model_path': str(OBJ_DET_OUTPUT_DIR / 'yolov8_disease_detection/weights/best.pt'),
    'config_path': str(yolo_yaml_path),
    'detection_visualization': str(OBJ_DET_OUTPUT_DIR / 'sample_detection.png')
}

# Save summary
with open(OBJ_DET_OUTPUT_DIR / 'model_summary.json', 'w') as f:
    json.dump(obj_det_summary, f, indent=2)

print("\nObject Detection Model Training Summary:")
print("="*70)
print(f"Model Type: {obj_det_summary['model_type']}")
print(f"Task: {obj_det_summary['task']}")
print(f"Disease Classes: {obj_det_summary['num_classes']}")
print(f"Training Images: {obj_det_summary['training_images']}")
print(f"Validation Images: {obj_det_summary['validation_images']}")
print(f"Total Annotations: {obj_det_summary['total_annotations']}")
print(f"\nSaved Model: {obj_det_summary['model_path']}")
print(f"Config File: {obj_det_summary['config_path']}")
print("="*70)
print(f"\nOutputs saved to: {OBJ_DET_OUTPUT_DIR}")